# Exercise: Implementing a Simple Continuous Batching Scheduler

Welcome! In this exercise, you'll build a simplified version of a scheduler, a core component of high-throughput LLM serving systems like vLLM. You'll get hands-on experience with the logic behind continuous batching.

In this exercise, you will:
- Implement the logic for queueing new inference requests.
- Write a function to free KV cache blocks from completed requests.
- Build the core scheduling algorithm that selects which requests to run based on available KV cache.

Let's get started!

In [ ]:
import enum
from collections import deque
from dataclasses import dataclass, field
from typing import List, Deque, Tuple

# --- Provided Helper Code (Do Not Modify) ---

class SequenceStatus(enum.Enum):
    """Enumeration for the status of a sequence."""
    WAITING = 1
    RUNNING = 2
    FINISHED = 3

@dataclass
class Sequence:
    """Represents a single request's state during its lifecycle."""
    request_id: str
    tokens: List[int]
    status: SequenceStatus = SequenceStatus.WAITING
    kv_blocks: List[int] = field(default_factory=list)

    def __repr__(self):
        return f"Seq(id={self.request_id}, status={self.status.name}, len={len(self.tokens)}, blocks={len(self.kv_blocks)})"

class SimulatedKVCacheManager:
    """A simplified KV cache block manager."""
    def __init__(self, total_blocks: int):
        self._total_blocks = total_blocks
        self._free_blocks = list(range(total_blocks))

    def get_num_free_blocks(self) -> int:
        return len(self._free_blocks)

    def allocate(self, num_blocks: int) -> List[int]:
        if num_blocks > self.get_num_free_blocks():
            raise ValueError("Not enough free blocks to allocate.")
        allocated = self._free_blocks[:num_blocks]
        self._free_blocks = self._free_blocks[num_blocks:]
        return allocated

    def free(self, blocks: List[int]):
        # In a real system, you'd need to handle fragmentation. Here, we keep it simple.
        self._free_blocks.extend(blocks)

class MockExecutor:
    """A mock executor that simulates one step of token generation."""
    @staticmethod
    def execute(batch: List[Sequence]) -> List[Sequence]:
        print(f"  Executor: Running a batch of {len(batch)} sequences.")
        for seq in batch:
            if seq.status == SequenceStatus.RUNNING:
                # Simulate generating one new token
                seq.tokens.append(seq.tokens[-1] + 1)
                # In a real system, you'd need a new KV block for this new token.
                # We'll handle block allocation in the scheduler's main loop.
                if len(seq.tokens) >= 10: # Arbitrary stop condition
                    seq.status = SequenceStatus.FINISHED
                    print(f"  Executor: Sequence {seq.request_id} finished.")
        return batch

# --- Main Class for the Exercise ---

class SimpleScheduler:
    """
    Manages waiting and running queues, and decides which sequences to run
    in the next iteration based on KV cache availability.
    """
    def __init__(self, kv_cache_manager: SimulatedKVCacheManager):
        self.kv_cache_manager = kv_cache_manager
        self.waiting_queue: Deque[Sequence] = deque()
        self.running_queue: List[Sequence] = []
        self._request_counter = 0

    def add_new_request(self, prompt_tokens: List[int]):
        """Public method to add a new request for scheduling."""
        request_id = f"req_{self._request_counter}"
        self._request_counter += 1
        self._add_sequence(Sequence(request_id, prompt_tokens))

    ### EXERCISE 1, 2, 3 WILL BE IMPLEMENTED HERE ###

    def step(self) -> List[Sequence]:
        """
        Performs one scheduling and execution step.
        This function is provided for you and orchestrates the methods you'll implement.
        """
        print(f"\n--- Scheduler Step ---")
        print(f"Start | Waiting: {len(self.waiting_queue)}, Running: {len(self.running_queue)}, Free Blocks: {self.kv_cache_manager.get_num_free_blocks()}")

        # 1. Free resources from any finished sequences from the previous step.
        self._free_finished_sequences()

        # 2. Try to schedule new sequences from the waiting queue.
        newly_scheduled = self._schedule_waiting_sequences()
        self.running_queue.extend(newly_scheduled)

        # 3. Ensure running sequences have enough blocks for the *next* token.
        #    This is a key part of continuous batching.
        for seq in self.running_queue:
            if seq.status == SequenceStatus.RUNNING:
                 # If the number of tokens is equal to allocated blocks, we need one more.
                if len(seq.tokens) == len(seq.kv_blocks):
                    if self.kv_cache_manager.get_num_free_blocks() >= 1:
                        new_block = self.kv_cache_manager.allocate(1)
                        seq.kv_blocks.extend(new_block)
                    else:
                        # Preemption would happen here in a more advanced scheduler.
                        # For now, we'll just stall.
                        print(f"  Scheduler: Stalling! No block for seq {seq.request_id} to grow.")
                        return [] # Return empty batch, nothing can run.

        print(f"Post-Schedule | Waiting: {len(self.waiting_queue)}, Running: {len(self.running_queue)}, Free Blocks: {self.kv_cache_manager.get_num_free_blocks()}")

        # 4. Return the final batch to be executed.
        return self.running_queue

### Exercise 1: Add Sequence to Waiting Queue

Your first task is to implement the `_add_sequence` method. This method is responsible for taking a new `Sequence` object and adding it to the scheduler's `waiting_queue`. The `waiting_queue` is a `deque` (double-ended queue), which is perfect for FIFO (First-In, First-Out) scheduling.

**Hint:** The `collections.deque` object has an `append` method, just like a standard Python list.

In [ ]:
def _add_sequence(self, seq: Sequence):
    """
    Adds a new sequence to the waiting queue.

    Args:
        seq (Sequence): The sequence object to add.
    """
    # Check that the sequence is in the correct initial state.
    assert seq.status == SequenceStatus.WAITING
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Add the implemented method to our class
SimpleScheduler._add_sequence = _add_sequence

In [ ]:
# Test for Exercise 1

print("--- Testing Exercise 1: _add_sequence ---")
kv_manager = SimulatedKVCacheManager(total_blocks=100)
scheduler = SimpleScheduler(kv_manager)

# Add a new request
seq1 = Sequence(request_id="req_1", tokens=[10, 20, 30])
scheduler._add_sequence(seq1)

assert len(scheduler.waiting_queue) == 1, f"Expected 1 sequence in waiting_queue, but got {len(scheduler.waiting_queue)}"
assert scheduler.waiting_queue[0].request_id == "req_1", "The wrong sequence was added to the queue."
assert scheduler.running_queue == [], "running_queue should be empty."

# Add another request
seq2 = Sequence(request_id="req_2", tokens=[5, 15])
scheduler._add_sequence(seq2)
assert len(scheduler.waiting_queue) == 2, f"Expected 2 sequences in waiting_queue, but got {len(scheduler.waiting_queue)}"
assert scheduler.waiting_queue[1].request_id == "req_2", "The second sequence was not added correctly."

print("✅ All tests for Exercise 1 passed!")

### Exercise 2: Free Finished Sequences

Great job! Now, let's handle completed requests. Before scheduling new sequences, the scheduler must "clean up" by identifying any sequences in the `running_queue` that have finished generating tokens.

Your task is to implement `_free_finished_sequences`. This method should:
1.  Iterate through the `running_queue`.
2.  For any sequence with a status of `FINISHED`, free its associated KV cache blocks using `self.kv_cache_manager.free()`.
3.  Remove all finished sequences from the `running_queue`.

**Hint:** It's often safer to build a new list of items to keep, rather than removing items from a list while you are iterating over it.

In [ ]:
def _free_finished_sequences(self):
    """
    Finds finished sequences in the running queue, frees their KV blocks,
    and removes them from the queue.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Add the implemented method to our class
SimpleScheduler._free_finished_sequences = _free_finished_sequences

In [ ]:
# Test for Exercise 2

print("\n--- Testing Exercise 2: _free_finished_sequences ---")
kv_manager = SimulatedKVCacheManager(total_blocks=50)
scheduler = SimpleScheduler(kv_manager)

# Manually set up the running queue with some finished sequences
seq_running = Sequence("running_1", [1,2,3], SequenceStatus.RUNNING, kv_blocks=[1,2,3])
seq_finished_1 = Sequence("finished_1", [4,5,6,7], SequenceStatus.FINISHED, kv_blocks=[4,5,6,7])
seq_finished_2 = Sequence("finished_2", [8,9], SequenceStatus.FINISHED, kv_blocks=[8,9])

scheduler.running_queue = [seq_running, seq_finished_1, seq_finished_2]

# Manually "allocate" the blocks so we can check if they are freed
_ = kv_manager.allocate(3 + 4 + 2)
initial_free_blocks = kv_manager.get_num_free_blocks()
assert initial_free_blocks == 50 - 9

# Run the function to test
scheduler._free_finished_sequences()

# Check the running queue
assert len(scheduler.running_queue) == 1, f"Expected 1 sequence left in running_queue, but got {len(scheduler.running_queue)}"
assert scheduler.running_queue[0].request_id == "running_1", "The wrong sequence was removed."

# Check if blocks were freed
# We freed 4 blocks from finished_1 and 2 from finished_2
expected_free_blocks = initial_free_blocks + 4 + 2
current_free_blocks = kv_manager.get_num_free_blocks()
assert current_free_blocks == expected_free_blocks, f"Expected {expected_free_blocks} free blocks, but got {current_free_blocks}"

print("✅ All tests for Exercise 2 passed!")

### Exercise 3: Schedule Waiting Sequences

This is the core of our scheduler! Your goal is to move sequences from the `waiting_queue` to the `running_queue` if and only if there are enough free KV cache blocks for their initial prompt.

In vLLM, a KV cache block doesn't just store one token's state; it stores several. For our simplification, we'll assume **one block is needed per token in the initial prompt**.

Your task is to implement `_schedule_waiting_sequences`:
1.  Iterate through the `waiting_queue` in the order they were added (FIFO).
2.  For each sequence, check if the number of free blocks is greater than or equal to the number of tokens in its prompt.
3.  If there are enough blocks:
    -   Allocate the required blocks from the `kv_cache_manager`.
    -   Assign the allocated block IDs to the sequence's `kv_blocks` attribute.
    -   Change the sequence's status to `RUNNING`.
    -   Add the sequence to a list of `newly_scheduled` sequences.
4.  If there are not enough blocks for the current sequence, you cannot schedule it or any subsequent sequences in the queue. Stop checking and return what you have scheduled so far.
5.  Finally, remove the sequences you scheduled from the `waiting_queue`.

In [ ]:
def _schedule_waiting_sequences(self) -> List[Sequence]:
    """
    Schedules sequences from the waiting queue if there is enough KV cache.

    Returns:
        List[Sequence]: A list of the newly scheduled sequences.
    """
    newly_scheduled: List[Sequence] = []
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###
    return newly_scheduled
# Add the implemented method to our class
SimpleScheduler._schedule_waiting_sequences = _schedule_waiting_sequences

In [ ]:
# Test for Exercise 3
print("\n--- Testing Exercise 3: _schedule_waiting_sequences ---")

# Case 1: Enough space for the first, but not the second
kv_manager = SimulatedKVCacheManager(total_blocks=10)
scheduler = SimpleScheduler(kv_manager)
scheduler._add_sequence(Sequence("req_A", [1]*8)) # needs 8 blocks
scheduler._add_sequence(Sequence("req_B", [1]*5)) # needs 5 blocks

newly_scheduled = scheduler._schedule_waiting_sequences()

assert len(newly_scheduled) == 1, f"Test 1: Expected 1 new sequence, got {len(newly_scheduled)}"
assert newly_scheduled[0].request_id == "req_A", "Test 1: Wrong sequence was scheduled"
assert newly_scheduled[0].status == SequenceStatus.RUNNING, "Test 1: Scheduled sequence status should be RUNNING"
assert len(newly_scheduled[0].kv_blocks) == 8, "Test 1: Incorrect number of blocks allocated"
assert len(scheduler.waiting_queue) == 1, f"Test 1: Expected 1 sequence left in waiting queue, got {len(scheduler.waiting_queue)}"
assert scheduler.waiting_queue[0].request_id == "req_B", "Test 1: Wrong sequence left in waiting queue"
assert kv_manager.get_num_free_blocks() == 2, f"Test 1: Expected 2 free blocks remaining, got {kv_manager.get_num_free_blocks()}"

# Case 2: Enough space for both
kv_manager = SimulatedKVCacheManager(total_blocks=20)
scheduler = SimpleScheduler(kv_manager)
scheduler._add_sequence(Sequence("req_C", [1]*8))
scheduler._add_sequence(Sequence("req_D", [1]*5))

newly_scheduled = scheduler._schedule_waiting_sequences()
assert len(newly_scheduled) == 2, f"Test 2: Expected 2 new sequences, got {len(newly_scheduled)}"
assert len(scheduler.waiting_queue) == 0, f"Test 2: Waiting queue should be empty, got {len(scheduler.waiting_queue)}"
assert kv_manager.get_num_free_blocks() == 7, f"Test 2: Expected 7 free blocks, got {kv_manager.get_num_free_blocks()}"

print("✅ All tests for Exercise 3 passed!")

### (Optional) Challenge: Priority and Preemption

In our `SimpleScheduler`, a very long request that arrives first can cause many short requests to wait for a long time (this is known as head-of-line blocking).

Real-world schedulers like vLLM's have more advanced features:
- **Priority:** Requests can be assigned a priority (e.g., 'high', 'low'). The scheduler would then prioritize scheduling high-priority requests from the waiting queue, even if they arrived after low-priority ones.
- **Preemption:** If a high-priority request arrives but there isn't enough KV cache, the scheduler can "preempt" (pause) a running low-priority request. This involves copying its KV cache to main memory, freeing the blocks, and scheduling the high-priority request. The preempted request is put back in the running queue to be swapped back in later.

For a challenge, think about how you might modify the `SimpleScheduler` and `Sequence` classes to support preemption. What new states or attributes would a `Sequence` need? How would the `_schedule` logic change?

In [ ]:
# --- Integration Test ---
# This cell puts everything together to run a small simulation.
# No need to modify anything here, just run it to see your scheduler in action!

print("\n--- Running Full Simulation ---")

# 1. Setup
kv_manager = SimulatedKVCacheManager(total_blocks=20)
scheduler = SimpleScheduler(kv_manager)
executor = MockExecutor()

# 2. Define incoming requests and their arrival times
requests_to_add = {
    0: [[1, 2, 3, 4, 5, 6, 7]],                             # req_0 (7 tokens)
    1: [[10, 11, 12, 13, 14, 15, 16, 17, 18]],              # req_1 (9 tokens)
    2: [[20, 21, 22]],                                      # req_2 (3 tokens)
}
max_steps = 15
num_requests = sum(len(v) for v in requests_to_add.values())
finished_count = 0

# 3. Simulation Loop
for step in range(max_steps):
    print(f"\n>>>>>> TIME STEP {step} <<<<<<")

    # Add any new requests that have "arrived"
    if step in requests_to_add:
        for prompt in requests_to_add[step]:
            print(f"Time {step}: Adding new request with {len(prompt)} tokens.")
            scheduler.add_new_request(prompt)

    # Scheduler performs its logic
    batch_for_executor = scheduler.step()

    # Executor "runs" the model on the batch
    if batch_for_executor:
        updated_sequences = executor.execute(batch_for_executor)
        # In a real system, the scheduler gets the updated state back
        scheduler.running_queue = updated_sequences
    else:
        print("  Executor: Received an empty batch. Nothing to run.")

    finished_this_step = sum(1 for seq in scheduler.running_queue if seq.status == SequenceStatus.FINISHED)
    if finished_this_step > 0:
        finished_count += finished_this_step

    if finished_count == num_requests:
        print(f"\nSimulation finished in {step+1} steps!")
        break

assert finished_count == num_requests, f"Simulation ended but not all requests finished. Completed: {finished_count}/{num_requests}"
print("✅ Full simulation completed successfully!")